## Data Analysis - Arbitration

In [14]:
#Permute: df['B'] = np.random.permutation(df['B'])

In [1]:
import matplotlib.pyplot as plt
from pandas.api.types import CategoricalDtype
import pandas as pd
import numpy as np
import os

In [2]:
'''
Starting by reading in the csv and converting it to a pandas df
'''
meanReachesIdx = pd.read_csv("./meanCSVfull.csv")

print(meanReachesIdx.columns)

Index(['Unnamed: 0', 'session', 'stim', 'maxV_idx', 'maxS_idx', 'maxF_idx',
       'minF_idx', 'maxE_idx', 'minE_idx'],
      dtype='object')


In [3]:
print(f"Max V: {max(meanReachesIdx['maxV_idx'])} Min V: {min(meanReachesIdx['maxV_idx'])}\n")

print(f"Difference between max and min V is: {max(meanReachesIdx['maxV_idx']) - min(meanReachesIdx['maxV_idx'])}")

Max V: 697 Min V: 23

Difference between max and min V is: 674


In [4]:
meanReachesIdx.iloc[2]

Unnamed: 0         2
session       210421
stim               0
maxV_idx         401
maxS_idx         132
maxF_idx         483
minF_idx           0
maxE_idx          20
minE_idx         334
Name: 2, dtype: int64

### Statistics Before Permutation:

In [5]:
'''
Calculating the distances between each idx and V and S, creating a dataframe from it, and saving it into a csv document
'''
calculatedDistances = []

for meanReach in meanReachesIdx.itertuples():
    max_firing_rate_dist_V = abs(meanReach[4] - meanReach[6])
    min_firing_rate_dist_V = abs(meanReach[4] - meanReach[7])

    max_energy_dist_V = abs(meanReach[4] - meanReach[8])
    min_energy_dist_V = abs(meanReach[4] - meanReach[9])


    calculatedDistances.append({
        "session" : meanReach[2],
        "stim" : meanReach[3],
        
        "Max Firing Rate Distance from V" : max_firing_rate_dist_V,
        "Min Firing Rate Distance from V" : min_firing_rate_dist_V,
        "Max Energy Distance from V" : max_energy_dist_V,
        "Min Energy Distance from V" : min_energy_dist_V,
    })

distancesDF = pd.DataFrame(calculatedDistances)

pd.DataFrame(distancesDF).to_csv("meanDistancesCSVfull.csv")


In [6]:
distancesDF['MinEnergy'] = distancesDF[['Max Energy Distance from V', 'Min Energy Distance from V']].min(axis=1)
distancesDF['MinFR'] = distancesDF[['Max Firing Rate Distance from V', 'Min Firing Rate Distance from V']].min(axis=1)

distancesDF['winnerE'] = distancesDF['MinEnergy'] < distancesDF['MinFR']
distancesDF['winnerF'] = distancesDF['MinFR'] < distancesDF['MinEnergy']

In [7]:
print(f"precent winner Energy {100* distancesDF[distancesDF['winnerE'] == 1].shape[0] / distancesDF.shape[0]:.2f}%")
print(f"precent winner FR     {100* distancesDF[distancesDF['winnerF'] == 1].shape[0] / distancesDF.shape[0]:.2f}%")
print("\n stim 0")
print(f"precent winner Energy {100* distancesDF[(distancesDF['winnerE'] == 1) & (distancesDF['stim'] == 0)].shape[0] / distancesDF[distancesDF['stim'] == 0].shape[0]:.2f}%")
print(f"precent winner FR {100* distancesDF[(distancesDF['winnerF'] == 1) & (distancesDF['stim'] == 0)].shape[0] / distancesDF[distancesDF['stim'] == 0].shape[0]:.2f}%")

print("\n stim 1")
print(f"precent winner Energy {100* distancesDF[(distancesDF['winnerE'] == 1) & (distancesDF['stim'] == 1)].shape[0] / distancesDF[distancesDF['stim'] == 1].shape[0]:.2f}%")
print(f"precent winner FR {100* distancesDF[(distancesDF['winnerF'] == 1) & (distancesDF['stim'] == 1)].shape[0] / distancesDF[distancesDF['stim'] == 1].shape[0]:.2f}%")

print("\n stim 2")
print(f"precent winner Energy {100* distancesDF[(distancesDF['winnerE'] == 1) & (distancesDF['stim'] == 2)].shape[0] / distancesDF[distancesDF['stim'] == 2].shape[0]:.2f}%")
print(f"precent winner FR {100* distancesDF[(distancesDF['winnerF'] == 1) & (distancesDF['stim'] == 2)].shape[0] / distancesDF[distancesDF['stim'] == 2].shape[0]:.2f}%")

precent winner Energy 34.11%
precent winner FR     64.77%

 stim 0
precent winner Energy 36.03%
precent winner FR 63.10%

 stim 1
precent winner Energy 30.25%
precent winner FR 68.25%

 stim 2
precent winner Energy 35.81%
precent winner FR 63.17%


In [37]:
distancesDF[]

SyntaxError: invalid syntax (1803741721.py, line 1)

In [12]:
### Referenced AI for this portion of code ###

target_cols = [
    'Max Energy Distance from V', 
    'Min Energy Distance from V',
    'Max Firing Rate Distance from V', 
    'Min Firing Rate Distance from V'
]
short_names = ["MaxE", "MinE", "MaxF", "MinF"]

# Creating a True/False matrix
mask = distancesDF[target_cols].values == distancesDF[target_cols].min(axis=1).values[:, None]

# 2. Join short_names where mask is True
# This handles ties like "MaxE & MinF" automatically
winners = [
    " & ".join([short_names[i] for i, is_match in enumerate(row) if is_match])
    for row in mask
]

# Adding a column to the dataframe and defining a categorical type to it
distancesDF['DistWinner'] = winners
unique_outcomes = sorted(distancesDF['DistWinner'].unique())
dist_cat_type = CategoricalDtype(categories=unique_outcomes, ordered=True)
distancesDF['DistWinner'] = distancesDF['DistWinner'].astype(dist_cat_type)

total_trials = len(distancesDF)

# Getting win counts (Inclusive of ties)
print("--- Win Counts (Inclusive of Ties) ---")
for name in short_names:
    # .str.contains finds the name even if it's part of a tie like "MaxE & MinF"
    count = distancesDF['DistWinner'].str.contains(name).sum()
    pctV = (count / total_trials) * 100
    print(f"{name}: {count} ({pctV:.2f}%)")

tEnergyWins = distancesDF['DistWinner'].str.contains("E").sum()
tFrWins = distancesDF['DistWinner'].str.contains("F").sum()
pctEnergy = (tEnergyWins/total_trials) * 100
pctFr = (tFrWins/total_trials) * 100

print(f"\n\nTotal Energy Wins: ({pctEnergy:.2f}%)\nTotal Firing Rate Wins: {tFrWins} ({pctFr:.2f}%)")

# Number of ties
total_ties = distancesDF['DistWinner'].str.contains(" & ").sum()

#the elements with ties and what they are
tied_summary = distancesDF.loc[distancesDF['DistWinner'].str.contains(" & "), ['session', 'DistWinner']]

print("\nTies:")
print(tied_summary)


--- Win Counts (Inclusive of Ties) ---
MaxE: 14 (35.90%)
MinE: 7 (17.95%)
MaxF: 9 (23.08%)
MinF: 10 (25.64%)


Total Energy Wins: (53.85%)
Total Firing Rate Wins: 19 (48.72%)

Ties:
    session   DistWinner
34   220518  MaxE & MaxF


### Statistics after Permutations

In [68]:
#Permute: df['B'] = np.random.permutation(df['B'])

In [32]:
'''
Calculating the distances between each idx and V and S, creating a dataframe from it, and saving it into a csv document
'''
calculatedDistances = []

permutatedReaches = meanReachesIdx

permutatedReaches['maxV_idx'] = np.random.permutation(meanReachesIdx['maxV_idx'])

for meanReach in permutatedReaches.itertuples():
    max_firing_rate_dist_V = abs(meanReach[4] - meanReach[6])
    min_firing_rate_dist_V = abs(meanReach[4] - meanReach[7])

    max_energy_dist_V = abs(meanReach[4] - meanReach[8])
    min_energy_dist_V = abs(meanReach[4] - meanReach[9])


    calculatedDistances.append({
        "session" : meanReach[2],
        "stim" : meanReach[3],
        
        "Max Firing Rate Distance from V" : max_firing_rate_dist_V,
        "Min Firing Rate Distance from V" : min_firing_rate_dist_V,
        "Max Energy Distance from V" : max_energy_dist_V,
        "Min Energy Distance from V" : min_energy_dist_V,
    })

distancesDF = pd.DataFrame(calculatedDistances)


In [33]:
### Referenced AI for this portion of code ###

target_cols = [
    'Max Energy Distance from V', 
    'Min Energy Distance from V',
    'Max Firing Rate Distance from V', 
    'Min Firing Rate Distance from V'
]
short_names = ["MaxE", "MinE", "MaxF", "MinF"]

# Creating a True/False matrix
mask = distancesDF[target_cols].values == distancesDF[target_cols].min(axis=1).values[:, None]

# 2. Join short_names where mask is True
# This handles ties like "MaxE & MinF" automatically
winners = [
    " & ".join([short_names[i] for i, is_match in enumerate(row) if is_match])
    for row in mask
]

# Adding a column to the dataframe and defining a categorical type to it
distancesDF['DistWinner'] = winners
unique_outcomes = sorted(distancesDF['DistWinner'].unique())
dist_cat_type = CategoricalDtype(categories=unique_outcomes, ordered=True)
distancesDF['DistWinner'] = distancesDF['DistWinner'].astype(dist_cat_type)

total_trials = len(distancesDF)

# Getting win counts (Inclusive of ties)
print("--- Win Counts (Inclusive of Ties) ---")
for name in short_names:
    # .str.contains finds the name even if it's part of a tie like "MaxE & MinF"
    count = distancesDF['DistWinner'].str.contains(name).sum()
    pctV = (count / total_trials) * 100
    print(f"{name}: {count} ({pctV:.2f}%)")

tEnergyWins = distancesDF['DistWinner'].str.contains("E").sum()
tFrWins = distancesDF['DistWinner'].str.contains("F").sum()
pctEnergy = (tEnergyWins/total_trials) * 100
pctFr = (tFrWins/total_trials) * 100

print(f"\n\nTotal Energy Wins: ({pctEnergy:.2f}%)\nTotal Firing Rate Wins: {tFrWins} ({pctFr:.2f}%)")

# Number of ties
total_ties = distancesDF['DistWinner'].str.contains(" & ").sum()

#the elements with ties and what they are
tied_summary = distancesDF.loc[distancesDF['DistWinner'].str.contains(" & "), ['session', 'DistWinner']]

print("\nTies:")
print(tied_summary)


--- Win Counts (Inclusive of Ties) ---
MaxE: 13 (33.33%)
MinE: 8 (20.51%)
MaxF: 9 (23.08%)
MinF: 10 (25.64%)


Total Energy Wins: (53.85%)
Total Firing Rate Wins: 19 (48.72%)

Ties:
    session   DistWinner
29   220516  MaxE & MaxF
